# 04 · Modelagem do Data Warehouse (esquema estrela)

Organizar os dados para análise:

- **Dimensão** = quem/o quê/onde/quando -> cliente, produto, data...
Gerado uma **chave substituta** `customer_key` além do ID original do Olist
- **Fato** = o que aconteceu
Uma tabela de fatos aponta para as dimensões pelas chaves substitutas e guarda as métricas (preço, frete, tempo de entrega)

Ter dimensões pequenas e reutilizáveis + fatos enxutos para consultas de negócio e dashboard rápidos e simples de escrever

In [1]:
# Configuracao inicial
import os
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")


def find_project_root():
    current = Path.cwd().resolve()
    for p in [current] + list(current.parents):
        if (p / "src").exists() and (p / "data").exists() and (p / "config").exists():
            return p
    raise RuntimeError("Raiz do projeto nao encontrada")


def project_path(*segments):
    return find_project_root().joinpath(*segments)


root = find_project_root()
os.chdir(root)
print(f"Diretorio de trabalho: {root}")


Diretorio de trabalho: C:\Users\user\Downloads\Códigos\olist-ecommerce-pipeline\Template


## O modelo

```
                              dim_date          dim_customers
                                 │                    │
dim_products ──▶ fact_order_items ◀── dim_sellers     │
                                 │                    │
                                 └──────▶ fact_orders ◀┘
                     fact_payments   fact_reviews
```

**Dimensões:** `dim_customers`, `dim_sellers`, `dim_products`, `dim_geolocation`, `dim_date`.

**Fatos**, cada um com uma granularidade específica
- `fact_order_items` grão: **item do pedido** (o mais fino, para análises de produto/categoria)
- `fact_orders` grão: **pedido**, já com métricas de entrega calculadas (dias até entregar, atraso)
- `fact_payments` grão: **registro de pagamento** (um pedido pode ter mais de um registro)
- `fact_reviews` grão: **avaliação**

Obs.: recria as tabelas do zero a cada execução (`init_schemas()` roda o DDL `sql/dw/create_dw_schema.sql`)

In [2]:
from src.etl.db import get_engine
from src.etl.dw_builder import DWBuilder

engine = get_engine()
builder = DWBuilder(engine)
builder.init_schemas()  # (re)cria as tabelas do schema dw

### Populando dimensões

Cada `build_dim_*` lê a tabela `staging.stg_*` correspondente e grava em `dw.dim_*`. `dim_date` é diferente:
não vem de nenhuma tabela staging, é uma "régua de calendário" gerada por código, cobrindo todo o período
dos pedidos.

In [3]:
builder.build_all_dimensions()

{'dim_customers': 99441,
 'dim_sellers': 3095,
 'dim_products': 32951,
 'dim_geolocation': 19015,
 'dim_date': 800}

### Populando fatos

As fatos dependem das dimensões já carregadas: para cada linha, o builder busca a chave substituta certa
(ex.: `customer_id` do Olist → `customer_key` do banco) e é essa chave que fica gravada na fato.

In [4]:
builder.build_all_facts()

{'fact_order_items': 112650,
 'fact_orders': 99441,
 'fact_payments': 103886,
 'fact_reviews': 99224}

### Conferindo contagens

In [5]:
for table in ["dim_customers", "dim_sellers", "dim_products", "dim_geolocation", "dim_date", "fact_order_items", "fact_orders", "fact_payments", "fact_reviews"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM dw.{table}", engine)["n"][0]
    print(f"{table}: {count}")

dim_customers: 99441
dim_sellers: 3095
dim_products: 32951
dim_geolocation: 19015
dim_date: 800
fact_order_items: 112650
fact_orders: 99441
fact_payments: 103886
fact_reviews: 99224
